# **read data**

In [1]:
import numpy as np
import pandas as pd
import sys

In [3]:
data = pd.read_csv("/content/Result_17.csv")

In [4]:
data.head(2)

,YEAR,WEEK,TIMERANGE,LEG,DEPTIME,ARRTIME,DEPDATE,LF
0,2026,25,04 - 06,DADSGN,545,720,2026-06-16,5
1,2026,29,20 - 22,SGNHUI,2110,2240,2026-07-15,0


In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 152277 entries, 0 to 152276
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   YEAR       152277 non-null  int64 
 1   WEEK       152277 non-null  int64 
 2   TIMERANGE  152277 non-null  object
 3   LEG        152277 non-null  object
 4   DEPTIME    152277 non-null  int64 
 5   ARRTIME    152277 non-null  int64 
 6   DEPDATE    152277 non-null  object
 7   LF         152277 non-null  int64 
dtypes: int64(5), object(3)
memory usage: 9.3+ MB


# **function**

In [7]:
# ham chuyen doi hhmm sang tong so phut
def time_to_minutes(time):
    hours = int(time[:2])
    minutes = int(time[2:])
    return hours * 60 + minutes

In [8]:
# ham chuyen so phut sang format hhmm
def minutes_to_time(minute):
    hours = int(minute // 60)
    minutes = int(minute % 60)
    return f"{hours:02d}{minutes:02d}"

# **ham nhan input tu nguoi dung**

In [9]:
def input_info(data):
  print("Type 'quit' or 'exit' at any prompt to stop the program.")

  # nhan thong tin year, leg, loai hanh trinh tu nguoi dung, bao loi va dung chuong trinh neu thong tin nhap vao khong hop le
  # nhap thong tin year
  year_str = input("Enter year: ")
  if year_str.lower() in ['quit', 'exit']:
    print("Exiting program.")
    return None
  try:
    year = int(year_str)
  except ValueError:
    print(f"Error: Year '{year_str}' is not a valid number.")
    return None

  # nhap thong tin loai hanh trinh, chi chap nhan 2 gia tri inbound/outbound
  while True:
    leg_type = input("Enter leg type (inbound/outbound): ")
    if leg_type.lower() in ['quit', 'exit']:
      print("Exiting program.")
      return None
    if leg_type.lower() in ['inbound', 'outbound']:
      break
    else:
      print("Leg type must be 'inbound' or 'outbound'. Please try again.")

  # nhap thong tin leg
  leg = input("Enter leg: ")
  if leg.lower() in ['quit', 'exit']:
    print("Exiting program.")
    return None

  # print thong tin khung gio cua leg, year vua nhap
  time_range_options = data[(data['YEAR'] == year) & (data['LEG'] == leg)]['TIMERANGE'].unique()
  if len(time_range_options) == 0:
    print(f"Error: No data found for Leg '{leg}', Year {year}.")
    return None
  print(f"Time range for leg '{leg}' of year {year}: {np.sort(time_range_options)}")

  # nhap thong tin time range
  time_range_str = input("Enter time range: ")
  if time_range_str.lower() in ['quit', 'exit']:
    print("Exiting program.")
    return None
  time_range = time_range_str

  # filter the data and return the required columns as a DataFrame
  filtered_data = data[(data['YEAR'] == year) & (data['LEG'] == leg) & (data['TIMERANGE'] == time_range)]
  # raise error if filtered_data empty
  if filtered_data.empty:
    print(f"Error: No data found for Leg '{leg}', Year {year}, Time Range '{time_range}'.")
    return None

  # select and return the desired columns, using existing 'LF'
  return_df = filtered_data[['YEAR', 'WEEK', 'LEG', 'DEPTIME', 'ARRTIME', 'TIMERANGE', 'LF']].copy()

  # avg arr/dep time based on leg_type
  if leg_type.lower() == 'inbound':
    return_df['Time_Str'] = return_df['ARRTIME'].astype(str).str.zfill(4)
    return_df['Time_Mean'] = return_df['Time_Str'].apply(time_to_minutes)
    time_column_name_for_output = 'AvgArrTime'
  else:
    return_df['Time_Str'] = return_df['DEPTIME'].astype(str).str.zfill(4)
    return_df['Time_Mean'] = return_df['Time_Str'].apply(time_to_minutes)
    time_column_name_for_output = 'AvgDepTime'

  # groupby and calculate mean for LF and Avg_Minutes
  result = return_df.groupby(['YEAR', 'WEEK', 'LEG', 'TIMERANGE']).agg(LF = ('LF', 'mean'), Avg_Minutes = ('Time_Mean', 'mean')).reset_index()

  # tinh avg time
  result[time_column_name_for_output] = result['Avg_Minutes'].apply(minutes_to_time)

  # chon cot hien thi o output
  input_df = result[['YEAR', 'WEEK', 'LEG', 'TIMERANGE', 'LF', time_column_name_for_output]]
  #print(input_df)
  return input_df, leg_type

# **ham thuc thi neu nguoi dung chon loai hanh trinh inbound**

In [10]:
def inbound(leg_input, year, time_range):
  outbound_legs_data = []
  target_arrival_airport = leg_input[-3:]

  # input data (no longer filtering by specific week, gets all weeks for the selected year, leg, time_range)
  original_leg_data = data[(data['YEAR'] == year) & (data['LEG'] == leg_input) & (data['TIMERANGE'] == time_range)].copy()

  # format arrival time cua leg input
  original_leg_data['Time_Str'] = original_leg_data['ARRTIME'].astype(str).str.zfill(4)
  original_leg_data['Time_Min'] = original_leg_data['Time_Str'].apply(time_to_minutes)
  # tinh avg arrival time PER WEEK
  original_leg_weekly_arrival_time = original_leg_data.groupby('WEEK')['Time_Min'].mean().reset_index()
  original_leg_weekly_arrival_time = original_leg_weekly_arrival_time.rename(columns={'Time_Min': 'original_leg_avg_arrival_time'})

  # filter year (already contains all weeks)
  filtered_context_data = data[(data['YEAR'] == year)].copy()

  # filter cac leg co 3 ki tu dau giong 3 ki tu cuoi cua leg input
  potential_outbound_legs = filtered_context_data[filtered_context_data['LEG'].str.startswith(target_arrival_airport)].copy()

  if potential_outbound_legs.empty:
    return pd.DataFrame(columns=['YEAR', 'WEEK', 'LEG', 'TIMERANGE', 'LF', 'DURATION', 'DEPTIME'])

  # deptime theo tong so phut
  potential_outbound_legs['DEPTIME_Str'] = potential_outbound_legs['DEPTIME'].astype(str).str.zfill(4)
  potential_outbound_legs['DEPTIME_Min'] = potential_outbound_legs['DEPTIME_Str'].apply(time_to_minutes)

  # group by leg, time range, WEEK de tinh LF, avg deptime
  grouped_outbound = potential_outbound_legs.groupby(['YEAR', 'WEEK', 'LEG', 'TIMERANGE']).agg(LF = ('LF', 'mean'), avg_deptime_min = ('DEPTIME_Min', 'mean')).reset_index()

  # tinh duration = avg deptime - avg arrival time (now merged by week)
  grouped_outbound = pd.merge(grouped_outbound, original_leg_weekly_arrival_time, on='WEEK', how='left')
  grouped_outbound['DURATION_in_minutes'] = grouped_outbound['avg_deptime_min'] - grouped_outbound['original_leg_avg_arrival_time']
  grouped_outbound['DURATION_in_minutes'] = grouped_outbound['DURATION_in_minutes'].apply(lambda x: x + 24 * 60 if x < 0 else x)
  grouped_outbound['DURATION_in_minutes'] = grouped_outbound['DURATION_in_minutes'].fillna(0) # fill NaN with 0
  grouped_outbound['DURATION'] = grouped_outbound['DURATION_in_minutes'].apply(minutes_to_time)

  # quy doi avg deptime ve format hhmm
  grouped_outbound['DEPTIME'] = grouped_outbound['avg_deptime_min'].apply(minutes_to_time)

  # output
  result_df = grouped_outbound[['YEAR', 'WEEK', 'LEG', 'TIMERANGE', 'LF', 'DURATION', 'DEPTIME']].copy()
  result_df = result_df.sort_values(by = ['YEAR', 'WEEK', 'LEG', 'TIMERANGE'], ascending = True).reset_index(drop = True)

  return result_df

# **ham thuc thi neu nguoi dung chon loai hanh trinh outbound**

In [11]:
def outbound(leg_input, year, time_range):
  target_departure_airport = leg_input[:3]

  # input data (no longer filtering by specific week, gets all weeks for the selected year, leg, time_range)
  original_leg_data = data[(data['YEAR'] == year) & (data['LEG'] == leg_input) & (data['TIMERANGE'] == time_range)].copy()

  original_leg_data['Time_Str'] = original_leg_data['DEPTIME'].astype(str).str.zfill(4)
  original_leg_data['Time_Min'] = original_leg_data['Time_Str'].apply(time_to_minutes)
  # tinh avg deptime cua leg input PER WEEK
  original_leg_weekly_deptime = original_leg_data.groupby('WEEK')['Time_Min'].mean().reset_index()
  original_leg_weekly_deptime = original_leg_weekly_deptime.rename(columns={'Time_Min': 'original_leg_avg_deptime'})

  # filter year (already contains all weeks)
  filtered_context_data = data[(data['YEAR'] == year)].copy()

  potential_inbound_legs = filtered_context_data[filtered_context_data['LEG'].str.endswith(target_departure_airport)].copy()

  if potential_inbound_legs.empty:
    return pd.DataFrame(columns = ['YEAR', 'WEEK', 'LEG', 'TIMERANGE', 'LF', 'DURATION', 'ARRTIME'])

  potential_inbound_legs['ARRTIME_Str'] = potential_inbound_legs['ARRTIME'].astype(str).str.zfill(4)
  potential_inbound_legs['ARRTIME_Min'] = potential_inbound_legs['ARRTIME_Str'].apply(time_to_minutes)

  # group by theo leg, time range, WEEK de tinh total bkg, cap, LF
  grouped_inbound = potential_inbound_legs.groupby(['YEAR', 'WEEK', 'LEG', 'TIMERANGE']).agg(LF = ('LF', 'mean'), avg_arrtime_min = ('ARRTIME_Min', 'mean')).reset_index()

  # tinh duration = avg deptime leg input - avg arrival time current leg
  grouped_inbound = pd.merge(grouped_inbound, original_leg_weekly_deptime, on='WEEK', how='left')
  grouped_inbound['DURATION_in_minutes'] = grouped_inbound['original_leg_avg_deptime'] - grouped_inbound['avg_arrtime_min']
  grouped_inbound['DURATION_in_minutes'] = grouped_inbound['DURATION_in_minutes'].apply(lambda x: x + 24 * 60 if x < 0 else x)
  grouped_inbound['DURATION_in_minutes'] = grouped_inbound['DURATION_in_minutes'].fillna(0)
  grouped_inbound['DURATION'] = grouped_inbound['DURATION_in_minutes'].apply(minutes_to_time)

  # quy doi arrival time ve format hhmm
  grouped_inbound['ARRTIME'] = grouped_inbound['avg_arrtime_min'].apply(minutes_to_time)

  # output
  result_df = grouped_inbound[['YEAR', 'WEEK', 'LEG', 'TIMERANGE', 'LF', 'DURATION', 'ARRTIME']].copy()
  result_df = result_df.sort_values(by=['YEAR', 'WEEK', 'LEG', 'TIMERANGE'], ascending=True).reset_index(drop=True)

  return result_df

# **output**

In [12]:
# goi ham input_info khi nguoi dung nhap thong tin
user_inputs = input_info(data)

if user_inputs is not None:
  df_input_info, leg_type_input = user_inputs
  leg_input = df_input_info['LEG'].iloc[0]
  year_input = df_input_info['YEAR'].iloc[0]
  time_range_input = df_input_info['TIMERANGE'].iloc[0]

  print("User Input Details:")
  display(df_input_info)

  # goi ham inbound/outbound tuy theo loai hanh trinh nguoi dung nhap
  if leg_type_input.lower() == 'inbound':
    result_df = inbound(leg_input, year_input, time_range_input)
    if not result_df.empty:
      print("\nInbound Legs:")
      display(result_df)
    else:
      print("No inbound legs found for the given input.")
  elif leg_type_input.lower() == 'outbound':
    result_df = outbound(leg_input, year_input, time_range_input)
    if not result_df.empty:
      print("\nOutbound Legs:")
      display(result_df)
    else:
      print("No outbound legs found for the given input.")
else:
  print("Input process terminated or invalid input provided. No further processing in this cell.")

Type 'quit' or 'exit' at any prompt to stop the program.
Enter year: 2026
Enter leg type (inbound/outbound): inbound
Enter leg: HANBKK
Time range for leg 'HANBKK' of year 2026: ['08 - 10' '12 - 14' '14 - 16' '16 - 18' '18 - 20']
Enter time range: 14 - 16
User Input Details:


,YEAR,WEEK,LEG,TIMERANGE,LF,AvgArrTime
0,2026,11,HANBKK,14 - 16,91.750000,1756
1,2026,12,HANBKK,14 - 16,82.375000,1755
2,2026,13,HANBKK,14 - 16,71.666667,1755
3,2026,43,HANBKK,14 - 16,44.000000,1755
4,2026,44,HANBKK,14 - 16,18.142857,1755
5,2026,45,HANBKK,14 - 16,9.142857,1755
6,2026,46,HANBKK,14 - 16,6.571429,1755
7,2026,47,HANBKK,14 - 16,9.571429,1755
8,2026,48,HANBKK,14 - 16,8.857143,1755
9,2026,49,HANBKK,14 - 16,6.000000,1755



Inbound Legs:


,YEAR,WEEK,LEG,TIMERANGE,LF,DURATION,DEPTIME
0,2026,11,BKKDAD,16 - 18,96.75,0003,1800
1,2026,11,BKKHAN,12 - 14,99.80,1823,1220
2,2026,11,BKKHAN,14 - 16,99.00,2158,1555
3,2026,11,BKKHAN,18 - 20,99.60,0107,1904
4,2026,11,BKKHAN,22 - 24,97.60,0413,2210
...,...,...,...,...,...,...,...
386,2026,53,BKKHAN,22 - 24,5.00,0415,2210
387,2026,53,BKKSGN,10 - 12,3.50,1725,1120
388,2026,53,BKKSGN,14 - 16,8.75,2025,1420
389,2026,53,BKKSGN,18 - 20,16.75,0140,1935


In [14]:
df_input_info = df_input_info.pivot(index = ['YEAR', 'LEG', 'TIMERANGE'], columns = 'WEEK', values = ['LF']).reset_index()
df_input_info

YEAR     LEG TIMERANGE     LF                                      \
WEEK                             11      12         13    43         44   
0     2026  HANBKK   14 - 16  91.75  82.375  71.666667  44.0  18.142857   

                                                                         \
WEEK        45        46        47        48   49         50         51   
0     9.142857  6.571429  9.571429  8.857143  6.0  17.428571  10.428571   

                     
WEEK        52   53  
0     3.142857  4.0

In [ ]:
# export df_input_info ra file csv
df_input_info.to_csv('input_info.csv', index=False)
# download file csv
import os
full_path = os.path.abspath('input_info.csv')
print(f"File path: {full_path}")

File path: c:\Users\hiennguyenthithanh\Downloads\input_info.csv


In [15]:
result_df = result_df.pivot(index = ['LEG', 'TIMERANGE', 'DEPTIME', 'DURATION'], columns = 'WEEK', values = ['LF']).reset_index()
result_df

LEG TIMERANGE DEPTIME DURATION     LF                        \
WEEK                                        11         12         13   
0     BKKDAD   16 - 18    1800     0003  96.75        NaN        NaN   
1     BKKDAD   16 - 18    1800     0004    NaN  92.666667        NaN   
2     BKKDAD   16 - 18    1800     0005    NaN        NaN  82.400000   
3     BKKDAD   18 - 20    1805     0000    NaN        NaN        NaN   
4     BKKDAD   18 - 20    1805     0010    NaN        NaN  68.000000   
5     BKKHAN   10 - 12    1155     0000    NaN        NaN        NaN   
6     BKKHAN   10 - 12    1155     1800    NaN        NaN  88.000000   
7     BKKHAN   12 - 14    1220     1823  99.80        NaN        NaN   
8     BKKHAN   12 - 14    1220     1824    NaN  96.285714        NaN   
9     BKKHAN   12 - 14    1220     1825    NaN        NaN  92.500000   
10    BKKHAN   14 - 16    1555     0000    NaN        NaN        NaN   
11    BKKHAN   14 - 16    1555     2158  99.00        NaN        NaN   
12    BKKHAN   14 - 16    1555     2159    NaN  92.571429        NaN   
13    BKKHAN   14 - 16    1555     2200    NaN        NaN  88.714286   
14    BKKHAN   18 - 20    1900     0104    NaN  92.571429        NaN   
15    BKKHAN   18 - 20    1900     0105    NaN        NaN  85.428571   
16    BKKHAN   18 - 20    1904     0107  99.60        NaN        NaN   
17    BKKHAN   18 - 20    1904     0109    NaN        NaN        NaN   
18    BKKHAN   18 - 20    1905     0000    NaN        NaN        NaN   
19    BKKHAN   22 - 24    2210     0413  97.60        NaN        NaN   
20    BKKHAN   22 - 24    2210     0414    NaN  88.857143        NaN   
21    BKKHAN   22 - 24    2210     0415    NaN        NaN  74.571429   
22    BKKHAN   22 - 24    2214     0419    NaN        NaN        NaN   
23    BKKHAN   22 - 24    2215     0000    NaN        NaN        NaN   
24    BKKSGN   10 - 12    1120     0000    NaN        NaN        NaN   
25    BKKSGN   10 - 12    1120     1724    NaN  93.000000        NaN   
26    BKKSGN   10 - 12    1120     1725    NaN        NaN  85.000000   
27    BKKSGN   10 - 12    1125     1728  99.20        NaN        NaN   
28    BKKSGN   14 - 16    1420     2023  96.40        NaN        NaN   
29    BKKSGN   14 - 16    1420     2024    NaN  93.571429        NaN   
30    BKKSGN   14 - 16    1420     2025    NaN        NaN  86.142857   
31    BKKSGN   14 - 16    1424     2029    NaN        NaN        NaN   
32    BKKSGN   14 - 16    1425     0000    NaN        NaN        NaN   
33    BKKSGN   18 - 20    1930     0000    NaN        NaN        NaN   
34    BKKSGN   18 - 20    1930     0135    NaN        NaN        NaN   
35    BKKSGN   18 - 20    1934     0139    NaN        NaN  80.857143   
36    BKKSGN   18 - 20    1935     0138  95.80        NaN        NaN   
37    BKKSGN   18 - 20    1935     0139    NaN  88.714286        NaN   
38    BKKSGN   18 - 20    1935     0140    NaN        NaN        NaN   
39    BKKSGN   20 - 22    2105     0000    NaN        NaN        NaN   
40    BKKSGN   20 - 22    2106     0311    NaN        NaN        NaN   
41    BKKSGN   20 - 22    2113     0318    NaN        NaN  67.428571   
42    BKKSGN   20 - 22    2115     0319    NaN  84.285714        NaN   
43    BKKSGN   20 - 22    2115     0320    NaN        NaN        NaN   
44    BKKSGN   20 - 22    2124     0327  90.00        NaN        NaN   

                                       ...                                \
WEEK         14         15         16  ...        44        45        46   
0           NaN        NaN        NaN  ...       NaN       NaN       NaN   
1           NaN        NaN        NaN  ...       NaN       NaN       NaN   
2           NaN        NaN        NaN  ...  1.833333  2.000000  1.500000   
3     74.857143  62.285714  53.571429  ...       NaN       NaN       NaN   
4           NaN        NaN        NaN  ...       NaN       NaN       NaN   
5     89.285714  88.714286  77.857143  ...       NaN       NaN       NaN   
6           NaN 

In [ ]:
# export result_df ra file csv
result_df.to_csv('result_df.csv', index=False)
# download file csv
import os
full_path = os.path.abspath('result_df.csv')
print(f"File path: {full_path}")

File path: c:\Users\hiennguyenthithanh\Downloads\result_df.csv
